# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the dataset schema.

In [ ]:
# List all record sets and their fields with @id references
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"RecordSet: {rs['@id']} | Name: {rs['name']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            print(f"  Field: {field.get('@id', '<no-id>')} | Name: {field.get('name', '<no-name>')}")
        else:
            print(f"  Field: {field}")
    print()

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use record set and field `@id`s from above.

In [ ]:
# Typical datasets have only one main record set, but we generalize to all record sets found
import collections

# Get the @id of each main record set
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Use dataset.records(record_set=<@id>) to extract each table/record set
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for RecordSet {record_set_id} with shape {df.shape}")
        else:
            print(f"No records in record set {record_set_id}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Show the columns of the first main record set as an example
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Columns of record set {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
    # Show a preview
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data to prepare for further analysis.

In [ ]:
import numpy as np

# Use the main record set for demonstration
df = dataframes[main_record_set_id]
# List numeric columns heuristically
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric fields found: {numeric_fields}")

# If no numeric columns were auto-inferred, try to coerce
if not numeric_fields:
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"After coercion, numeric fields: {numeric_fields}")

# Select a numeric field for analysis (choose the first one)
if numeric_fields:
    numeric_field = numeric_fields[0]
    threshold = df[numeric_field].quantile(0.75) # Use 75th percentile as example threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (75th percentile):\n")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a likely categorical field if available
    potential_group_fields = [col for col in df.columns if col not in numeric_fields]
    group_field = None
    for col in potential_group_fields:
        if df[col].nunique() < len(df) // 5:
            group_field = col
            break
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped statistics by {group_field} (showing means):")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # If a group_field was found, boxplot by group_field
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook walked through loading metadata, extracting and exploring the main tables within the FAIR^2 mass spectrometry dataset. With the `mlcroissant` library, you can reference dataset structures via their Croissant `@id`s for robust, FAIR-aligned workflows. Continue analysis or integrate this notebook with training/validation tasks as needed.